# Traveling-Wave Discovery (Re=300)

Goal: find novel traveling-wave solutions by fuzzing TWModel seeds, deduplicating by
wave speeds + norms + shear, and then promoting unique candidates to a full Channelflow
`findsoln` run.

This notebook is designed to handle multiple symmetry groups and discretizations.

Notes:
- The default guess strategy is random. You can enable shear-targeted guesses or
  trajectory-sampled guesses by toggling `guess_strategy` below.
- The `findsoln` stage will write results to `out_dir`.


In [1]:
import Pkg; Pkg.activate("../../.")

  Activating project at `~/dev/MyCloudAtlas.jl`


In [2]:
using CloudAtlas
using LinearAlgebra
using Statistics
using Random
using Dates
using DelimitedFiles
using Serialization
using Base.Threads
using ChannelflowWrapper


## Configuration


In [3]:
# Domain sizes (α = 2π/Lx, γ = 2π/Lz)
# Choose these to match the target Channelflow DNS box.
α, γ = 2π/6.0, 2π/4.0

Re = 300.0

# Discretizations: (J, K, L)
discretizations = [
    (2, 4, 7),
    (3, 5, 7),
    # (3, 5, 9),
]

# Symmetry groups to explore
sx, sy, sz, tx, tz = CloudAtlas.halfbox_symmetries()

symmetry_groups = [
    (
        name = "txtz",
        H = [tx * tz],
        symm_file = joinpath(@__DIR__, "txtz.asc"),
    ),
    (
        name = "sxtx",
        H = [sx * tx],
        symm_file = joinpath(@__DIR__, "sxtx.asc"),
    ),
    (
        name = "sztx",
        H = [sz * tx],
        symm_file = joinpath(@__DIR__, "sztx.asc"),
    ),
    (
        name = "sxtz",
        H = [sx * tz],
        symm_file = joinpath(@__DIR__, "sxtz.asc"),
    ),
    (
        name = "sztz",
        H = [sz * tz],
        symm_file = joinpath(@__DIR__, "sztz.asc"),
    ),
    (
        name = "sxytxz",
        H = [(sx * sy) * (tx * tz)],
        symm_file = joinpath(@__DIR__, "sxytxz.asc"),
    ),
    (
        name = "tx",
        H = [tx],
        symm_file = joinpath(@__DIR__, "tx.asc"),
    ),
    (
        name = "tz",
        H = [tz],
        symm_file = joinpath(@__DIR__, "tz.asc"),
    ),
]

# Attempts per symmetry/discretization
attempts_per_level = 10_000

# Hookstep parameters
hookparams = SearchParams(
    ftol = 1e-8,
    xtol = 1e-10,
    δ = 0.02,
    Nnewton = 20,
    Nhook = 4,
    Nmusearch = 6,
    verbosity = 0,
)

# Guess strategy options: :random, :shear_target, :trajectory
guess_strategy = :random
xnorm = 0.4
shear_target = 1.2
shear_tol = 0.05

# Dedup tolerances
fp_tol = (
    cx = 1e-3,
    cz = 1e-3,
    nm = 2e-2,
    shear = 2e-2,
)

# Accept/reject thresholds
norm_threshold = 1e-3
speed_threshold = 1e-5

# Channelflow promotion settings
T = 10.0
out_dir = joinpath(@__DIR__, "tw_discovery_re300")
mkpath(out_dir)

# Reference field for Channelflow conversions
reference_path = joinpath(@__DIR__, "TW1-2pi1piRe200-40x49x40.nc")
reference_field_converted = joinpath(out_dir, "reference_field_$(α)_$(γ).nc")


"/home/ebenq/dev/MyCloudAtlas.jl/notebooks/tw_discovery/tw_discovery_re300/reference_field_1.0471975511965976_1.5707963267948966.nc"

## Helper types and functions


In [4]:
struct SolutionFingerprint
    cx::Float64
    cz::Float64
    nm::Float64
    shear::Float64
end

function fingerprint(model, ξ)
    x, cx, cz = extract_components(ξ, model)
    return SolutionFingerprint(cx, cz, norm(x), shear(x, model))
end

function is_distinct(new_fp::SolutionFingerprint, archive::Vector{SolutionFingerprint}; tol = fp_tol)
    for fp in archive
        if isapprox(new_fp.cx, fp.cx, atol = tol.cx) &&
           isapprox(new_fp.cz, fp.cz, atol = tol.cz) &&
           isapprox(new_fp.nm, fp.nm, atol = tol.nm) &&
           isapprox(new_fp.shear, fp.shear, atol = tol.shear)
            return false
        end
    end
    return true
end

function random_guess(model, rng; xnorm = 0.4)
    m = length(model)
    x = randn(rng, m)
    x = xnorm / norm(x) * x
    cx = randn(rng) * 0.1
    cz = randn(rng) * 0.1
    return [x; cx; cz]
end

function shear_target_guess(model, rng; xnorm = 0.4, target = 1.2, tol = 0.05, max_tries = 100)
    for _ in 1:max_tries
        ξ = random_guess(model, rng; xnorm = xnorm)
        x = ξ[1:length(model)]
        if abs(shear(x, model) - target) <= tol
            return ξ
        end
    end
    # fallback
    return random_guess(model, rng; xnorm = xnorm)
end

function build_guess(model, rng; strategy = :random, xnorm = 0.4, target = 1.2, tol = 0.05)
    if strategy == :random
        return random_guess(model, rng; xnorm = xnorm)
    elseif strategy == :shear_target
        return shear_target_guess(model, rng; xnorm = xnorm, target = target, tol = tol)
    else
        error("Unknown strategy: $(strategy)")
    end
end

function save_summary(path, solutions, model)
    header = ["id" "cx" "cz" "norm" "shear"]
    if isempty(solutions)
        writedlm(path, header, ',')
        return
    end

    rows = Matrix{Float64}(undef, length(solutions), 5)
    for (i, ξ) in enumerate(solutions)
        x, cx, cz = extract_components(ξ, model)
        rows[i, 1] = i
        rows[i, 2] = cx
        rows[i, 3] = cz
        rows[i, 4] = norm(x)
        rows[i, 5] = shear(x, model)
    end

    writedlm(path, vcat(header, rows), ',')
end



save_summary (generic function with 1 method)

## Fuzzing + deduplication


In [5]:
function fuzz_tw_solutions(model::TWModel, Re::Real;
    n_attempts = 1000,
    xnorm = 0.4,
    strategy = :random,
    shear_target = 1.2,
    shear_tol = 0.05,
    hookparams = SearchParams(),
    norm_threshold = 1e-3,
    speed_threshold = 1e-5,
)
    m = length(model)
    solutions = Vector{Vector{Float64}}()
    fingerprints = Vector{SolutionFingerprint}()

    data_lock = ReentrantLock()
    io_lock = ReentrantLock()

    rngs = [MersenneTwister(0xC0FFEE + i) for i in 1:Threads.maxthreadid()]
    progress = Threads.Atomic{Int}(0)
    progress_every = max(1, n_attempts ÷ 100)

    @threads for attempt in 1:n_attempts
        tid = threadid()
        rng = tid <= length(rngs) ? rngs[tid] : Random.default_rng()
        ξ_guess = build_guess(model, rng;
            strategy = strategy,
            xnorm = xnorm,
            target = shear_target,
            tol = shear_tol,
        )

        f(ξ) = model.g(ξ, Re)
        Df(ξ) = model.Dg(ξ, Re)

        ξ_star, converged = CloudAtlas.hookstepsolve(f, Df, ξ_guess, hookparams)

        if converged
            x, cx, cz = extract_components(ξ_star, model)
            if norm(x) > norm_threshold && (abs(cx) > speed_threshold || abs(cz) > speed_threshold)
                fp = fingerprint(model, ξ_star)
                lock(data_lock) do
                    if is_distinct(fp, fingerprints)
                        push!(fingerprints, fp)
                        push!(solutions, ξ_star)
                        lock(io_lock) do
                            println("[Thread $(threadid())] Unique TW: cx=$(round(fp.cx, digits=4)), cz=$(round(fp.cz, digits=4)), |x|=$(round(fp.nm, digits=4)), shear=$(round(fp.shear, digits=4))")
                        end
                    end
                end
            end
        end

        done = Threads.atomic_add!(progress, 1)
        if done % progress_every == 0 || done == n_attempts
            lock(io_lock) do
                pct = round(100 * done / n_attempts; digits=1)
                println("Progress: $(done)/$(n_attempts) ($(pct)%) (unique=$(length(solutions)))")
            end
        end
    end

    println("Done. Found $(length(solutions)) unique solutions.")
    return solutions, fingerprints
end


fuzz_tw_solutions (generic function with 1 method)

## Channelflow promotion


In [6]:
function ensure_reference_field(reference_path, reference_field_converted; α, γ)
    if !isfile(reference_field_converted)
        changegrid(reference_path, reference_field_converted; al = α, ga = γ)
    end
    return reference_field_converted
end

function promote_with_findsoln!(solutions, model, Re;
    symm_file,
    out_dir,
    reference_field_converted,
    T = 10.0,
)
    m = length(model)
    mkpath(out_dir)
    for (idx, ξ) in enumerate(solutions)
        timestamp = Dates.format(now(), "MM-DD-HHMMSS")
        sol_dir = joinpath(out_dir, "sol_$(idx)_$(timestamp)")
        mkpath(sol_dir)

        guess_path = joinpath(sol_dir, "u_guess.nc")
        sigma_file = joinpath(sol_dir, "sigma.asc")

        coeff2field(ξ[1:m], model.ijkl, reference_field_converted, guess_path)
        save_sigma(model, ξ[end - 1], ξ[end], T, sigma_file)

        try
            findsoln(guess_path;
                R = Re,
                eqb = true,
                xrel = model.keep_cx,
                zrel = model.keep_cz,
                symms = abspath(symm_file),
                sigma = sigma_file,
                od = sol_dir,
                T = T,
            )
        catch e
            println("findsoln failed: $(e)")
        end
    end
end


promote_with_findsoln! (generic function with 1 method)

## Main sweep


In [7]:
for symm in symmetry_groups
    symm_name = symm.name
    H = symm.H
    symm_file = symm.symm_file

    println("\n=== Symmetry: $(symm_name) ===")
    model = ODEModel(α, γ, 1, 2, 2, H; normalize = false, tw = true);
    println("keep_cx = $(model.keep_cx)")
    println("keep_cz = $(model.keep_cz)")
end


=== Symmetry: txtz ===
J,K,L,m == 1,2,2,36
(2J+1)(2K+1)(2L+1) + 1 == 76
Making matrices B,A1,A2,S3...
Making quadratic operator N...
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 
Making matrices Cx,Cz...
Phase constraints: keep_cx = true, keep_cz = true
keep_cx = true
keep_cz = true

=== Symmetry: sxtx ===
J,K,L,m == 1,2,2,36
(2J+1)(2K+1)(2L+1) + 1 == 76
Making matrices B,A1,A2,S3...
Making quadratic operator N...
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 
Making matrices Cx,Cz...
Phase constraints: keep_cx = false, keep_cz = true
keep_cx = false
keep_cz = true

=== Symmetry: sztx ===
J,K,L,m == 1,2,2,39
(2J+1)(2K+1)(2L+1) + 1 == 76
Making matrices B,A1,A2,S3...
Making quadratic operator N...
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 
Making matrices Cx,Cz...
Phase constraints: keep_cx = true, keep_cz = false
k

In [ ]:
reference_field_converted = ensure_reference_field(reference_path, reference_field_converted; α = α, γ = γ)
for (J, K, L) in discretizations
    for symm in symmetry_groups
        symm_name = symm.name
        H = symm.H
        symm_file = symm.symm_file
    
        println("\n=== Symmetry: $(symm_name) ===")
        println("symm_file: $(symm_file)")
        println("\n--- J,K,L = $(J),$(K),$(L) ---")

        model = ODEModel(α, γ, J, K, L, H; normalize = false, tw = true)

        level_dir = joinpath(out_dir, symm_name, "jkl_$(J)_$(K)_$(L)")
        mkpath(level_dir)

        solutions, fingerprints = fuzz_tw_solutions(
            model,
            Re;
            n_attempts = attempts_per_level,
            xnorm = xnorm,
            strategy = guess_strategy,
            shear_target = shear_target,
            shear_tol = shear_tol,
            hookparams = hookparams,
            norm_threshold = norm_threshold,
            speed_threshold = speed_threshold,
        )

        # Save summary of unique solutions
        summary_path = joinpath(level_dir, "solutions_summary.csv")
        save_summary(summary_path, solutions, model)

        # Save raw solutions (for later reuse)
        serialized_path = joinpath(level_dir, "solutions.bin")
        open(serialized_path, "w") do io
            serialize(io, solutions)
        end

        # Promote to Channelflow
        promote_with_findsoln!(
            solutions,
            model,
            Re;
            symm_file = symm_file,
            out_dir = joinpath(level_dir, "findsoln"),
            reference_field_converted = reference_field_converted,
            T = T,
        )
    end
end



=== Symmetry: txtz ===
symm_file: /home/ebenq/dev/MyCloudAtlas.jl/notebooks/tw_discovery/txtz.asc

--- J,K,L = 2,4,7 ---
J,K,L,m == 2,4,7,346
(2J+1)(2K+1)(2L+1) + 1 == 676
Making matrices B,A1,A2,S3...
Making quadratic operator N...
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 217 218 21

f^T: ....10
Newton iteration number 0
Current state of Newton iteration:
   fcount_newton   == 1
   fcount_optimiza == 0
   L2Norm(x)       == 0.133286
   L2Norm(dxN)     == 0
   L2Norm(dxOpt)   == 0
   L2Dist(x,x0)    == 0
gx == L2Norm(G(x)) : 
   initial  gx == 0.0121015
   previous gx == 0.0121015
   current  gx == 0.0121015
rx == 1/2 L2Norm2(G(x)) : 
   initial  rx == 7.32236e-05
   previous rx == 7.32236e-05
   current  rx == 7.32236e-05
         delta == 0.01
Newt,GMRES == 0,0, f^T: ....10 res == 0.868531
Newt,GMRES == 0,1, f^T: ....10 res == 0.865007
Newt,GMRES == 0,2, f^T: ....10 res == 0.819198
Newt,GMRES == 0,3, f^T: ....10 res == 0.764887
Newt,GMRES == 0,4, f^T: ....10 res == 0.759334
Newt,GMRES == 0,5, f^T: ....10 res == 0.75699
Newt,GMRES == 0,6, f^T: ....10 res == 0.722981
Newt,GMRES == 0,7, f^T: ....10 res == 0.705256
Newt,GMRES == 0,8, f^T: ....10 res == 0.510555
Newt,GMRES == 0,9, f^T: ....10 res == 0.461389
Newt,GMRES == 0,10, f^T: ....10 res == 0.444449
Newt,GMRES ==

f^T: ....10
Newton iteration number 0
Current state of Newton iteration:
   fcount_newton   == 1
   fcount_optimiza == 0
   L2Norm(x)       == 0.0573056
   L2Norm(dxN)     == 0
   L2Norm(dxOpt)   == 0
   L2Dist(x,x0)    == 0
gx == L2Norm(G(x)) : 
   initial  gx == 0.0063368
   previous gx == 0.0063368
   current  gx == 0.0063368
rx == 1/2 L2Norm2(G(x)) : 
   initial  rx == 2.00775e-05
   previous rx == 2.00775e-05
   current  rx == 2.00775e-05
         delta == 0.01
Newt,GMRES == 0,0, f^T: ....10 res == 0.990263
Newt,GMRES == 0,1, f^T: ....10 res == 0.964966
Newt,GMRES == 0,2, f^T: ....10 res == 0.664418
Newt,GMRES == 0,3, f^T: ....10 res == 0.39487
Newt,GMRES == 0,4, f^T: ....10 res == 0.346762
Newt,GMRES == 0,5, f^T: ....10 res == 0.291825
Newt,GMRES == 0,6, f^T: ....10 res == 0.290089
Newt,GMRES == 0,7, f^T: ....10 res == 0.224585
Newt,GMRES == 0,8, f^T: ....10 res == 0.141331
Newt,GMRES == 0,9, f^T: ....10 res == 0.104203
Newt,GMRES == 0,10, f^T: ....10 res == 0.0655083
Newt,GMRES 

f^T: ....10
Newton iteration number 0
Current state of Newton iteration:
   fcount_newton   == 1
   fcount_optimiza == 0
   L2Norm(x)       == 0.0573055
   L2Norm(dxN)     == 0
   L2Norm(dxOpt)   == 0
   L2Dist(x,x0)    == 0
gx == L2Norm(G(x)) : 
   initial  gx == 0.0063368
   previous gx == 0.0063368
   current  gx == 0.0063368
rx == 1/2 L2Norm2(G(x)) : 
   initial  rx == 2.00775e-05
   previous rx == 2.00775e-05
   current  rx == 2.00775e-05
         delta == 0.01
Newt,GMRES == 0,0, f^T: ....10 res == 0.990263
Newt,GMRES == 0,1, f^T: ....10 res == 0.964967
Newt,GMRES == 0,2, f^T: ....10 res == 0.664418
Newt,GMRES == 0,3, f^T: ....10 res == 0.394869
Newt,GMRES == 0,4, f^T: ....10 res == 0.346761
Newt,GMRES == 0,5, f^T: ....10 res == 0.291824
Newt,GMRES == 0,6, f^T: ....10 res == 0.290088
Newt,GMRES == 0,7, f^T: ....10 res == 0.224584
Newt,GMRES == 0,8, f^T: ....10 res == 0.141331
Newt,GMRES == 0,9, f^T: ....10 res == 0.104203
Newt,GMRES == 0,10, f^T: ....10 res == 0.0655082
Newt,GMRES

f^T: ....10
Newton iteration number 0
Current state of Newton iteration:
   fcount_newton   == 1
   fcount_optimiza == 0
   L2Norm(x)       == 0.0573056
   L2Norm(dxN)     == 0
   L2Norm(dxOpt)   == 0
   L2Dist(x,x0)    == 0
gx == L2Norm(G(x)) : 
   initial  gx == 0.0063368
   previous gx == 0.0063368
   current  gx == 0.0063368
rx == 1/2 L2Norm2(G(x)) : 
   initial  rx == 2.00775e-05
   previous rx == 2.00775e-05
   current  rx == 2.00775e-05
         delta == 0.01
Newt,GMRES == 0,0, f^T: ....10 res == 0.990263
Newt,GMRES == 0,1, f^T: ....10 res == 0.964966
Newt,GMRES == 0,2, f^T: ....10 res == 0.664418
Newt,GMRES == 0,3, f^T: ....10 res == 0.39487
Newt,GMRES == 0,4, f^T: ....10 res == 0.346762
Newt,GMRES == 0,5, f^T: ....10 res == 0.291825
Newt,GMRES == 0,6, f^T: ....10 res == 0.290089
Newt,GMRES == 0,7, f^T: ....10 res == 0.224585
Newt,GMRES == 0,8, f^T: ....10 res == 0.141331
Newt,GMRES == 0,9, f^T: ....10 res == 0.104203
Newt,GMRES == 0,10, f^T: ....10 res == 0.0655083
Newt,GMRES 

Excessive output truncated after 524309 bytes.